In [0]:
# 1. Τα στοιχεία σύνδεσης με το Azure
storage_account_name = "tzelas1dl"

# ΒΑΛΕ ΤΟ ΠΡΑΓΜΑΤΙΚΟ ΣΟΥ ΚΛΕΙΔΙ ΑΝΑΜΕΣΑ ΣΤΑ ΑΥΤΑΚΙΑ (Σβήσε το δικό μου κείμενο)
storage_account_key = "<YOUR_AZURE_STORAGE_KEY>"

container_name = "raw"

# 2. Ρύθμιση του Spark για το Azure Blob Storage (wasbs)
spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.blob.core.windows.net",
    storage_account_key
)

# 3. Φτιάχνουμε τα μονοπάτια (paths) για το πού βρίσκονται τα αρχεία
inventory_path = f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net/inventory_raw.csv"
suppliers_path = f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net/suppliers_data.csv"

# 4. Διαβάζουμε τα CSV χρησιμοποιώντας το PySpark
df_inventory = spark.read.csv(inventory_path, header=True, inferSchema=True)
df_suppliers = spark.read.csv(suppliers_path, header=True, inferSchema=True)

# 5. Εμφανίζουμε τα δεδομένα της αποθήκης!
display(df_inventory)

ProductID,ProductName,Quantity,LastRestockDate,SupplierID
PRD-012,Ρητίνη Ε,-30,2026-01-18,SUP-4
PRD-019,Διαλύτης Β,276,2026-03-11,SUP-5
PRD-035,Ρητίνη Ε,145,2026-01-27,SUP-2
PRD-048,Λιπαντικό Γ,379,2026-02-16,SUP-1
PRD-030,Ρητίνη Ε,-11,2026-02-02,SUP-4
PRD-015,Διαλύτης Β,146,2026-01-30,SUP-1
PRD-048,Διαλύτης Β,347,2026-02-13,SUP-5
PRD-030,Ρητίνη Ε,-10,2026-03-17,SUP-2
PRD-029,Διαλύτης Β,-41,2026-03-01,SUP-3
PRD-011,Χημικό Α,-10,2026-02-14,SUP-2


In [0]:
from pyspark.sql.functions import col

# 1. Καθαρισμός του πίνακα Αποθήκης (Inventory)
df_inventory_clean = (
    df_inventory
    .filter(col("Quantity") >= 0)           # Κρατάμε ΜΟΝΟ όσα έχουν θετικό απόθεμα (πετάμε τα αρνητικά)
    .dropna(subset=["LastRestockDate"])     # Πετάμε τις γραμμές που ξέχασαν να βάλουν ημερομηνία
    .dropDuplicates()                       # Πετάμε τις διπλοεγγραφές (ίδια ακριβώς γραμμή 2 φορές)
)

# 2. Καθαρισμός του πίνακα Προμηθευτών (Suppliers)
# Εδώ πετάμε τις διπλοεγγραφές με βάση το ID του προμηθευτή
df_suppliers_clean = df_suppliers.dropDuplicates(["SupplierID"])

# 3. Ένωση των δύο πινάκων (LEFT JOIN)
# Φέρνουμε τα στοιχεία του προμηθευτή δίπλα στο κάθε προϊόν της αποθήκης
df_silver = df_inventory_clean.join(df_suppliers_clean, on="SupplierID", how="left")

# 4. Εμφανίζουμε το καθαρό, ενοποιημένο αποτέλεσμα!
print("Πλήθος εγγραφών ΠΡΙΝ τον καθαρισμό:", df_inventory.count())
print("Πλήθος εγγραφών ΜΕΤΑ τον καθαρισμό:", df_silver.count())
display(df_silver)

Πλήθος εγγραφών ΠΡΙΝ τον καθαρισμό: 215
Πλήθος εγγραφών ΜΕΤΑ τον καθαρισμό: 93


SupplierID,ProductID,ProductName,Quantity,LastRestockDate,SupplierName,ContactEmail,LeadTimeDays
SUP-2,PRD-028,Καθαριστικό Δ,419,2026-01-19,Beta Logistics,info@beta.com,5
SUP-3,PRD-001,Διαλύτης Β,385,2026-02-18,Gamma Supplies,null,2
SUP-3,PRD-027,Λιπαντικό Γ,65,2026-02-10,Gamma Supplies,null,2
SUP-5,PRD-021,Λιπαντικό Γ,151,2026-03-05,Epsilon Materials,hello@epsilon.com,4
SUP-1,PRD-038,Διαλύτης Β,272,2026-02-20,Alpha Chemicals,contact@alpha.gr,3
SUP-2,PRD-025,Λιπαντικό Γ,89,2026-03-13,Beta Logistics,info@beta.com,5
SUP-1,PRD-035,Καθαριστικό Δ,105,2026-03-05,Alpha Chemicals,contact@alpha.gr,3
SUP-2,PRD-001,Καθαριστικό Δ,22,2026-01-18,Beta Logistics,info@beta.com,5
SUP-1,PRD-022,Ρητίνη Ε,35,2026-02-22,Alpha Chemicals,contact@alpha.gr,3
SUP-4,PRD-027,Λιπαντικό Γ,22,2026-03-06,Delta Lubes,sales@delta.gr,7


In [0]:
import pyspark.sql.functions as F

# 1. Φτιάχνουμε το Gold Layer (Συγκεντρωτικά δεδομένα)
df_gold = (
    df_silver
    .groupBy("ProductID", "ProductName")
    .agg(
        F.sum("Quantity").alias("Total_Inventory_Quantity"), # Αθροίζουμε τα αποθέματα
        F.count("Quantity").alias("Number_Of_Restocks")      # Μετράμε πόσες φορές παραλάβαμε
    )
    .orderBy("Total_Inventory_Quantity") # Ταξινομούμε από το μικρότερο απόθεμα στο μεγαλύτερο
)

print("--- ΔΕΔΟΜΕΝΑ GOLD LAYER (ΕΤΟΙΜΑ ΓΙΑ POWER BI) ---")
display(df_gold)

# 2. Αποθήκευση του τελικού πίνακα πίσω στο Azure Blob Storage (στον φάκελο processed)
# Σημείωση: Χρησιμοποιούμε το storage_account_name που έχουμε ήδη ορίσει στο πρώτο κελί
output_path = f"wasbs://processed@{storage_account_name}.blob.core.windows.net/gold_inventory_report"

# Το coalesce(1) λέει στο Spark να βγάλει 1 ενιαίο αρχείο CSV (και όχι πολλά μικρά)
(
    df_gold.coalesce(1)
    .write.csv(output_path, header=True, mode="overwrite")
)

print(f"\nΕΠΙΤΥΧΙΑ! Το Gold Layer αποθηκεύτηκε στον φάκελο 'processed' στο Azure.")

--- ΔΕΔΟΜΕΝΑ GOLD LAYER (ΕΤΟΙΜΑ ΓΙΑ POWER BI) ---


ProductID,ProductName,Total_Inventory_Quantity,Number_Of_Restocks
PRD-043,Διαλύτης Β,10,1
PRD-005,Λιπαντικό Γ,12,1
PRD-037,Χημικό Α,14,1
PRD-003,Ρητίνη Ε,18,1
PRD-001,Καθαριστικό Δ,22,1
PRD-031,Καθαριστικό Δ,32,1
PRD-022,Ρητίνη Ε,35,1
PRD-026,Καθαριστικό Δ,42,1
PRD-027,Ρητίνη Ε,47,1
PRD-050,Λιπαντικό Γ,56,1



ΕΠΙΤΥΧΙΑ! Το Gold Layer αποθηκεύτηκε στον φάκελο 'processed' στο Azure.
